# كشكول التجارب: تطبيق إعادة الترتيب عملياً (المحاضرة 45)

أهلاً بيك في معمل المحاضرة 45!
في المحاضرة اللي فاتت فهمنا نظرياً إن البحث السريع (زي FAISS أو BM25) ممكن يجيب 10 مستندات، بس المستند الأهم ممكن يكون في المركز الرابع أو الخامس.
النهارده هنجرب عملياً إزاي نموذج **إعادة الترتيب (Reranker)** بيبص على السؤال والمستند مع بعض ويخلي المستند الأكثر دقة يقفز فوراً للمركز الأول!

### 1. تجهيز النماذج والأدوات
يلا نبدأ باستيراد النماذج وتجهيز نموذج التضمين ونموذج الـ Reranker:

In [1]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder
from flashrank import Ranker, RerankRequest
import time

load_dotenv()

# التضمين العادي للبحث السريع
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# نموذج إعادة الترتيب الدقيق
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# نموذج FlashRank الخفيف جداً
fast_ranker = Ranker()
print("تم تحميل جميع النماذج بنجاح!")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_10852\3742482126.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

تم تحميل جميع النماذج بنجاح!


### 2. تجهيز مجموعة وثائق تجريبية
تعال نكتب وثائق فيها كلمات متشابهة ظاهرياً علشان نشوف مين هيعرف يفرق:

In [2]:
sample_docs = [
    Document(page_content="لغة بايثون تدعم البرمجة كائنية التوجه وتتميز بسهولة قراءة الكود.", metadata={"id": "python_general"}),
    Document(page_content="لتسريع كود بايثون في الإنتاج نستخدم مكتبة NumPy وعمليات الحوسبة الموجهة.", metadata={"id": "python_speed"}),
    Document(page_content="محرك إعادة الترتيب يقوم بفحص السؤال والفقرة معاً لتقييم الصلة الدلالية بدقة شديدة.", metadata={"id": "reranker_core"}),
    Document(page_content="القطار المغناطيسي فائق السرعة في اليابان يسير بسرعة تتجاوز 300 كم/ساعة.", metadata={"id": "bullet_train"}),
]

vector_store = FAISS.from_documents(sample_docs, embeddings)
print("تم تخزين الوثائق في فهرس FAISS.")

INFO:faiss.loader:Loading faiss with AVX2 support.


INFO:faiss.loader:Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")


INFO:faiss.loader:Loading faiss.


INFO:faiss.loader:Successfully loaded faiss.


تم تخزين الوثائق في فهرس FAISS.


### 3. تجربة البحث السريع العادي
تعال نسأل سؤال عن تسريع الأداء ونشوف الترتيب الأولي:

In [3]:
query = "كيف أقوم بتسريع تنفيذ برامج بايثون والحصول على أعلى أداء؟"

# البحث الأولي السريع
initial_results = vector_store.similarity_search(query, k=3)

print("الترتيب الأولي عبر البحث الكثيف (FAISS):")
for i, doc in enumerate(initial_results, start=1):
    print(f"[{i}] {doc.page_content}")

الترتيب الأولي عبر البحث الكثيف (FAISS):
[1] لغة بايثون تدعم البرمجة كائنية التوجه وتتميز بسهولة قراءة الكود.
[2] القطار المغناطيسي فائق السرعة في اليابان يسير بسرعة تتجاوز 300 كم/ساعة.
[3] محرك إعادة الترتيب يقوم بفحص السؤال والفقرة معاً لتقييم الصلة الدلالية بدقة شديدة.


### 4. تطبيق نموذج Cross-Encoder لإعادة الترتيب
دلوقتي هنخلي الـ CrossEncoder يقيس درجات الصلة لكل مستند ونشوف الترتيب الجديد:

In [4]:
pairs = [(query, doc.page_content) for doc in initial_results]
scores = reranker.predict(pairs)

reranked = sorted(zip(initial_results, scores), key=lambda x: x[1], reverse=True)

print("الترتيب بعد تطبيق الـ Reranker:")
for i, (doc, score) in enumerate(reranked, start=1):
    print(f"المركز {i} (الدرجة: {score:.2f}) -> {doc.page_content}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

الترتيب بعد تطبيق الـ Reranker:
المركز 1 (الدرجة: 7.51) -> لغة بايثون تدعم البرمجة كائنية التوجه وتتميز بسهولة قراءة الكود.
المركز 2 (الدرجة: 7.46) -> القطار المغناطيسي فائق السرعة في اليابان يسير بسرعة تتجاوز 300 كم/ساعة.
المركز 3 (الدرجة: 7.26) -> محرك إعادة الترتيب يقوم بفحص السؤال والفقرة معاً لتقييم الصلة الدلالية بدقة شديدة.


### 5. تجربة مكتبة FlashRank الخفيفة السريعة
تعال نجرب نفس الفكرة مع FlashRank ونشوف النتيجة وسرعة التنفيذ:

In [5]:
passages = [{"id": d.metadata["id"], "text": d.page_content} for d in initial_results]
req = RerankRequest(query=query, passages=passages)

start = time.perf_counter()
fr_results = fast_ranker.rerank(req)
elapsed = (time.perf_counter() - start) * 1000

print(f"زمن تنفيذ FlashRank: {elapsed:.2f} ملي ثانية\n")
for item in fr_results:
    print(f"الدرجة: {item['score']:.4f} | {item['text']}")

زمن تنفيذ FlashRank: 3.20 ملي ثانية

الدرجة: 1.0000 | محرك إعادة الترتيب يقوم بفحص السؤال والفقرة معاً لتقييم الصلة الدلالية بدقة شديدة.
الدرجة: 1.0000 | القطار المغناطيسي فائق السرعة في اليابان يسير بسرعة تتجاوز 300 كم/ساعة.
الدرجة: 1.0000 | لغة بايثون تدعم البرمجة كائنية التوجه وتتميز بسهولة قراءة الكود.


### 6. جرب بنفسك!
تقدر تعدل السؤال فوق أو تضيف نصوص جديدة وتشوف إزاي درجات الـ Reranker بتعكس الفهم الحقيقي لمعنى السؤال.